# 00 — Convert upstream demo CSV files to HDF5

This optional notebook reproduces the bundled HDF5 demo fixtures. Normal users can skip it because the converted files are already tracked.

In [ ]:
import os
import sys
from pathlib import Path

import h5py
import numpy as np
import pandas as pd
from tqdm import tqdm

repo_root = Path.cwd().resolve()
if not (repo_root / "src").is_dir():
    repo_root = repo_root.parent
sys.path.insert(0, str(repo_root))
CSV_ROOT = repo_root / "data/csv"
H5_ROOT = repo_root / "data/h5"
DATASETS = ["dataA", "dataB"]  # <-- your folder names under data/csv/
MIN_CELLS = 5000  # samples with fewer cells than this are skipped

## Convert

In [ ]:
for dataset in DATASETS:
    csv_dir = CSV_ROOT / dataset
    h5_dir = H5_ROOT / dataset
    h5_dir.mkdir(parents=True, exist_ok=True)

    # Accept both plain and gzipped CSVs; pandas decompresses .csv.gz transparently.
    csv_files = sorted(
        f for f in os.listdir(csv_dir) if f.endswith((".csv", ".csv.gz"))
    )

    for csv_file in tqdm(csv_files, desc=f"Converting {dataset}"):
        data = pd.read_csv(csv_dir / csv_file)
        features = np.round(data.iloc[:, :-1].values, 2).astype(np.float32)

        if features.shape[0] < MIN_CELLS:
            tqdm.write(f"Skipping {csv_file}: only {features.shape[0]} rows")
            continue

        cell_types_encoded = np.array(
            [str(ct).encode("utf-8") for ct in data.iloc[:, -1]]
        )
        feature_names = np.array([c.encode("utf-8") for c in data.columns[:-1]])

        stem = csv_file[:-7] if csv_file.endswith(".csv.gz") else csv_file[:-4]
        out_path = h5_dir / f"{stem}.h5"
        with h5py.File(out_path, "w") as h5file:
            h5file.create_dataset(
                "data", data=features, compression="gzip", compression_opts=4
            )
            h5file.create_dataset(
                "cell_types",
                data=cell_types_encoded,
                compression="gzip",
                compression_opts=4,
            )
            h5file.create_dataset("feature_names", data=feature_names)

## Validate

In [ ]:
for dataset in DATASETS:
    h5_dir = H5_ROOT / dataset
    sample = sorted(os.listdir(h5_dir))[0]
    with h5py.File(h5_dir / sample, "r") as f:
        markers = [m.decode("utf-8") for m in f["feature_names"][:]]
        n_cells, n_markers = f["data"].shape
        unique_cts = sorted(set(ct.decode("utf-8") for ct in f["cell_types"][:]))
    print(f"{dataset}/{sample}: {n_cells} cells × {n_markers} markers")
    print(f"  markers: {markers}")
    print(f"  cell types ({len(unique_cts)}): {unique_cts}")

In [ ]:
import torch

from src.data.dataset import CyTOFDataset

dataset = CyTOFDataset(
    data_dirs=[str(H5_ROOT / "dataA"), str(H5_ROOT / "dataB")], subset_size=10000
)
features, cell_types, sample_name = dataset[0]
print(f"Sample: {sample_name}")
print(f"Features: {features.shape}, dtype={features.dtype}")
print(f"Cell types: {cell_types.shape}, n_unique={len(torch.unique(cell_types))}")
print(f"Shared markers ({len(dataset.shared_markers)}): {dataset.shared_markers}")